# PREPROCESAMIENTO

In [ ]:

%load_ext autoreload
%autoreload 2
from mic_array.patron import MicArray
pip install nbstripout


## CARGA DE ARCHIVOS DE AUDIO

Se puede cargar un array de micrófonos desde una carpeta con archivos de audio, siempre que los archivos sigan un formato de nombre específico para los mics del array y tambien la referencia. 

In [ ]:
# Formato 1: V es número de mic
ma = MicArray.from_audio(
    "data/media",
    array_pattern = "mic_{MIC}_ang_forte_{H}.wav",
    ref_pattern   = "mic_ref_ang_forte_{H}.wav",
)

# Formato 2: V es ángulo de elevación
# ma = MicArray.from_audio(
#     "data/audio/forte",
#     array_pattern = "mic_{V}_ang_forte_{H}.wav",
#     ref_pattern   = "mic_ref_ang_forte_{H}.wav",
# )

In [ ]:
# Ver qué tiene
print(ma.angles)      # → [0, 10, 20, ..., 180]
print(ma.thetas)  # → ['ref', 0, 10, ..., 180]
print(ma.tensor.shape)  # → (n_angles, n_thetas, n_samples)

# O directamente el array numpy
i_az = ma._az_to_row(0)
i_el = ma._th_to_col(0)
signal = ma.tensor[i_az, i_el, :]


# Graficos

Se puede graficar:
- Todas los azimuth para una elevación dada (con o sin envolvente)
- Todas las elevaciones para un azimuth dado (con o sin envolvente)
- Una toma específica (azimuth y elevación) con o sin envolvente

Se puede elegir mediante los atributos `downsampling_graph` y `smoothing_ms` si se quiere hacer un downsampling al momento de graficar indicando un valor numérico (ej: `downsampling_graph=10`) o un suavizado con un filtro de media movil dada en milisegundos (ej: `smoothing_ms=10`).

In [ ]:
# ma.downsampling_graph = 10   # toma 1 de cada 10 muestras (default)
# ma.downsampling_graph = 1    # sin downsampling, señal completa
ma.downsampling_graph = 100   # más agresivo, más rápido

Smoothing para el filtro de media movil que grafica mejor la envolvente a travves de la transformada de Hilbert

In [ ]:
# ma.smoothing_ms = 0    # sin suavizado
ma.smoothing_ms = 20   # suavizado 20ms (default)
# ma.smoothing_ms = 100  # más suave

In [ ]:
# Un solo audio (azimuth + theta)
ma.plot(azimuth=0, theta=90, envelope=True)  # envelope=True por defecto

# Todas las elevaciones de un azimut
ma.plot(azimuth=10, envelope=True)

# Todos los azimuts para una theta
ma.plot(theta=90, envelope=True)
ma.plot(theta='ref', envelope=True)

# Con título personalizado
ma.plot(azimuth=0, title="Titulo genérico")

# Filtrado de Bajas frecuencias con butterworth de 4° orden

In [ ]:
ma_filtered = ma.copy() # Se crea una copia para poder tener el array con los datos crudos siempre
ma_filtered.hpf(200)   # HPF 200 Hz, 4° orden Butterworth

Se verifica si realmente se aplico el filtrado

In [ ]:
ma.plot(azimuth=0, theta=50, envelope=True, dB=True, title="Audio sin filtrar")
ma_filtered.plot(azimuth=0, theta=50, dB=True, envelope=True, title="Con HPF 200 Hz")

# Alineamiento de micrófonos

Para alinear se debe utilizar la toma de referencia que es la que mejor SNR tiene, graficando su envolvente en dB se puede detectar el ruido (en dBFS) a partir de ello se puede tomar luego un umbral al momento de alinear ya que esto se hace por onset, es decir cuando la señal supera dicho umbral, ya que la GCC para este metodo entre distintas tomas no da muy bien al ser tomas distintas.

In [ ]:
ma_filtered.plot(theta='ref', dB=True, envelope=True)

Se hace una nueva copia de la información, aca se coloca como parametro donde se quiere que se dejen alineada las tomas del mic de referencia y tambien se determina a partir de que umbral debe hacerse la detección del onset.

In [ ]:
# Paso 1: alinear todos los ref al mismo punto
ma_aligned = ma_filtered.copy()
ma_aligned.align_takes(theta='ref', target_onset=0.5, threshold_dB=-55)

In [ ]:
ma_aligned.plot(theta='ref', envelope=True)

In [ ]:
ma_aligned_to_ref = ma_aligned.copy()
ma_aligned_to_ref.align_to_ref(theta='ref')

In [ ]:
ma_aligned_to_ref.plot(theta='ref', dB=True)

In [ ]:
for az in range(10, 190, 30):
    ma_aligned_to_ref.plot(azimuth=az, envelope=True, dB=True)

Guardar el tensor alineado para no tener que repetir el preprocesamiento cada vez que se quiera utiizar los datos

In [ ]:
ma_aligned_to_ref.save('data/tensores/forte_full_aligned.npz')